In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q torch_geometric

In [3]:
import os, sys
BASE = "/content/drive/MyDrive/cicidsposter"
for f in ("CICDS2017.csv", "AttackGraph.txt", "models.py"):
    assert os.path.exists(os.path.join(BASE, f)), f"not found: {os.path.join(BASE, f)}"
sys.path.append(BASE)
os.chdir(BASE)
print("working folder:", os.getcwd())

working folder: /content/drive/MyDrive/cicidsposter


### Libraries

In [4]:
print("Hello world")

Hello world


In [5]:
import re
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from torch_geometric.nn import GCNConv
from models import GCN_EW
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


### Attack Graph

In [7]:
path = os.path.join(BASE, "AttackGraph.txt")

In [8]:
with open(path, 'r') as f:
    content = f.read()

In [9]:
print(content[:500])

1,"canAccessHost(machine3)","OR",0
2,"RULE 8 (Access a host through executing code on the machine)","AND",0
3,"execCode(machine3,user)","OR",0
4,"RULE 0 (When a principal is compromised any machine he has an account on will also be compromised)","AND",0
4,1,-1
5,"hasAccount(victim,machine3,user)","LEAF",1
4,5,-1
6,"principalCompromised(victim)","OR",0
7,"RULE 11 (password sniffing)","AND",0
8,"execCode(machine1,root)","OR",0
9,"RULE 2 (remote exploit of a server program)","AND",0
10,"netAccess(m


In [10]:
edges = re.findall(r'^(\d+),(\d+),-1$', content, flags=re.MULTILINE)

In [11]:
edges

[('4', '1'),
 ('4', '5'),
 ('11', '12'),
 ('11', '13'),
 ('10', '11'),
 ('9', '10'),
 ('9', '14'),
 ('9', '15'),
 ('8', '9'),
 ('7', '8'),
 ('7', '16'),
 ('6', '7'),
 ('19', '20'),
 ('19', '18'),
 ('18', '19'),
 ('21', '20'),
 ('25', '18'),
 ('24', '25'),
 ('26', '22'),
 ('24', '26'),
 ('29', '30'),
 ('29', '8'),
 ('28', '29'),
 ('31', '30'),
 ('35', '8'),
 ('34', '35'),
 ('36', '32'),
 ('34', '36'),
 ('33', '34'),
 ('33', '16'),
 ('33', '6'),
 ('32', '33'),
 ('31', '32'),
 ('28', '31'),
 ('27', '28'),
 ('38', '39'),
 ('37', '38'),
 ('27', '37'),
 ('24', '27'),
 ('23', '24'),
 ('23', '40'),
 ('23', '6'),
 ('22', '23'),
 ('21', '22'),
 ('18', '21'),
 ('17', '18'),
 ('17', '40'),
 ('6', '17'),
 ('41', '32'),
 ('41', '16'),
 ('6', '41'),
 ('42', '22'),
 ('42', '40'),
 ('6', '42'),
 ('43', '3'),
 ('43', '5'),
 ('6', '43'),
 ('4', '6'),
 ('3', '4'),
 ('2', '3'),
 ('1', '2'),
 ('46', '47'),
 ('46', '8'),
 ('45', '46'),
 ('48', '47'),
 ('48', '32'),
 ('45', '48'),
 ('44', '45'),
 ('50', '51')

In [12]:
len(edges)

72

In [13]:
raw_nodes = re.findall(r'^(\d+),"([^"]+)",(?:"?([^",\s]+)"?),(\d+)$', content, flags=re.MULTILINE)

In [14]:
raw_nodes

[('1', 'canAccessHost(machine3)', 'OR', '0'),
 ('2',
  'RULE 8 (Access a host through executing code on the machine)',
  'AND',
  '0'),
 ('3', 'execCode(machine3,user)', 'OR', '0'),
 ('4',
  'RULE 0 (When a principal is compromised any machine he has an account on will also be compromised)',
  'AND',
  '0'),
 ('5', 'hasAccount(victim,machine3,user)', 'LEAF', '1'),
 ('6', 'principalCompromised(victim)', 'OR', '0'),
 ('7', 'RULE 11 (password sniffing)', 'AND', '0'),
 ('8', 'execCode(machine1,root)', 'OR', '0'),
 ('9', 'RULE 2 (remote exploit of a server program)', 'AND', '0'),
 ('10', 'netAccess(machine1,tcp,445)', 'OR', '0'),
 ('11', 'RULE 6 (direct network access)', 'AND', '0'),
 ('12', 'hacl(internet,machine1,tcp,445)', 'LEAF', '1'),
 ('13', 'attackerLocated(internet)', 'LEAF', '1'),
 ('14', 'networkServiceInfo(machine1,smbd,tcp,445,root)', 'LEAF', '1'),
 ('15',
  "vulExists(machine1,'CVE-2026-4408',smbd,remoteExploit,privEscalation)",
  'LEAF',
  '1'),
 ('16', 'hasAccount(victim,mach

In [15]:
node_properties = []

In [16]:
for item in raw_nodes:
    node_id = item[0]
    desc = item[1]
    node_type = item[2]
    val = item[3]

    if node_type == 'OR':
        shape = 'diamond'
    elif node_type == 'AND':
        shape = 'ellipse'
    else:
        shape = 'box'

    formatted_string = f'label="{node_id}:{desc}:{val}",shape={shape}'
    node_properties.append(formatted_string)

In [17]:
node_properties

['label="1:canAccessHost(machine3):0",shape=diamond',
 'label="2:RULE 8 (Access a host through executing code on the machine):0",shape=ellipse',
 'label="3:execCode(machine3,user):0",shape=diamond',
 'label="4:RULE 0 (When a principal is compromised any machine he has an account on will also be compromised):0",shape=ellipse',
 'label="5:hasAccount(victim,machine3,user):1",shape=box',
 'label="6:principalCompromised(victim):0",shape=diamond',
 'label="7:RULE 11 (password sniffing):0",shape=ellipse',
 'label="8:execCode(machine1,root):0",shape=diamond',
 'label="9:RULE 2 (remote exploit of a server program):0",shape=ellipse',
 'label="10:netAccess(machine1,tcp,445):0",shape=diamond',
 'label="11:RULE 6 (direct network access):0",shape=ellipse',
 'label="12:hacl(internet,machine1,tcp,445):1",shape=box',
 'label="13:attackerLocated(internet):1",shape=box',
 'label="14:networkServiceInfo(machine1,smbd,tcp,445,root):1",shape=box',
 'label="15:vulExists(machine1,\'CVE-2026-4408\',smbd,remoteE

In [18]:
len(node_properties)

51

In [19]:
node_dict={}
nodes = [item[0] for item in raw_nodes]

In [20]:
nodes

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '50',
 '51']

In [21]:
for item in node_properties:
    label_match=re.search(r'label="(.*)"',item)
    label_text=label_match.group(1)

    parts=label_text.split(':')
    node_id=nodes.index(parts[0])
    statement=parts[1]

    match=re.match(r'(.+)\((.*)\)',statement)
    predicate=match.group(1).strip()
    attr_text=match.group(2).strip()

    if ',' in attr_text:
        attributes=attr_text.split(',')
    else:
        attributes=attr_text.split()

    clean_attributes=[]
    for a in attributes:
        clean_attributes.append(a.strip("'\""))
    attributes=clean_attributes

    shape_match=re.search(r'shape=(\w+)',item)
    shape=shape_match.group(1)

    node_dict[node_id]={
        'predicate':predicate,
        'attributes':attributes,
        'shape':shape
    }

In [22]:
len(node_dict)

51

In [23]:
node_dict[0]

{'predicate': 'canAccessHost', 'attributes': ['machine3'], 'shape': 'diamond'}

### Encoding

In [24]:
word2idx={}
idx=0

In [25]:
for n in node_dict:
    node=node_dict[n]
    words=[node['predicate']]+node['attributes']

    for word in words:
        if word not in word2idx:
            word2idx[word]=idx
            idx+=1

In [26]:
len(word2idx)

73

In [27]:
word2idx

{'canAccessHost': 0,
 'machine3': 1,
 'RULE 8': 2,
 'Access': 3,
 'a': 4,
 'host': 5,
 'through': 6,
 'executing': 7,
 'code': 8,
 'on': 9,
 'the': 10,
 'machine': 11,
 'execCode': 12,
 'user': 13,
 'RULE 0': 14,
 'When': 15,
 'principal': 16,
 'is': 17,
 'compromised': 18,
 'any': 19,
 'he': 20,
 'has': 21,
 'an': 22,
 'account': 23,
 'will': 24,
 'also': 25,
 'be': 26,
 'hasAccount': 27,
 'victim': 28,
 'principalCompromised': 29,
 'RULE 11': 30,
 'password': 31,
 'sniffing': 32,
 'machine1': 33,
 'root': 34,
 'RULE 2': 35,
 'remote': 36,
 'exploit': 37,
 'of': 38,
 'server': 39,
 'program': 40,
 'netAccess': 41,
 'tcp': 42,
 '445': 43,
 'RULE 6': 44,
 'direct': 45,
 'network': 46,
 'access': 47,
 'hacl': 48,
 'internet': 49,
 'attackerLocated': 50,
 'networkServiceInfo': 51,
 'smbd': 52,
 'vulExists': 53,
 'CVE-2026-4408': 54,
 'remoteExploit': 55,
 'privEscalation': 56,
 'machine2': 57,
 'RULE 1': 58,
 'local': 59,
 'CVE-2021-3156': 60,
 'sudo': 61,
 'localExploit': 62,
 'RULE 9': 

In [28]:
num_nodes=len(node_dict)
vocab_size=len(word2idx)

In [29]:
node_features=np.zeros((num_nodes,vocab_size))

In [30]:
node_features

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(51, 73))

In [31]:
for node_id in node_dict:
    node=node_dict[node_id]
    words=[node['predicate']]+node['attributes']

    for word in words:
        col=word2idx[word]
        node_features[node_id][col]=1

In [32]:
node_features.shape

(51, 73)

In [33]:
node_features

array([[1., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 1., 0.]], shape=(51, 73))

### Extracting Privilege Nodes


In [34]:
action_node_idx=[]

In [35]:
for node_id in node_dict:
    node=node_dict[node_id]
    if node['shape']=='diamond':
        action_node_idx.append(node_id)

In [36]:
len(action_node_idx)

14

In [37]:
action_node_idx

[0, 2, 5, 7, 9, 17, 21, 23, 27, 31, 33, 36, 44, 48]

In [38]:
adj_matrix=np.zeros((num_nodes,num_nodes))

In [39]:
for edge in edges:
    child, parent = edge

    source_idx = nodes.index(parent)
    target_idx = nodes.index(child)

    adj_matrix[source_idx][target_idx] = 1

In [40]:
edge_index=np.array(np.nonzero(adj_matrix))

In [41]:
adj_matrix.shape

(51, 51)

In [42]:
edge_index.shape

(2, 72)

### Dataset


In [43]:
data = pd.read_csv(
    os.path.join(BASE, "CICDS2017.csv")
)

In [44]:
data.shape

(2830743, 79)

In [45]:
data.head

<bound method NDFrame.head of           Destination Port   Flow Duration   Total Fwd Packets  \
0                    54865               3                   2   
1                    55054             109                   1   
2                    55055              52                   1   
3                    46236              34                   1   
4                    54863               3                   2   
...                    ...             ...                 ...   
2830738                 53           32215                   4   
2830739                 53             324                   2   
2830740              58030              82                   2   
2830741                 53         1048635                   6   
2830742                 53           94939                   4   

          Total Backward Packets  Total Length of Fwd Packets  \
0                              0                           12   
1                              1                            6   
2                              1                            6   
3                              1                            6   
4                              0                           12   
...                          ...                          ...   
2830738                        2                          112   
2830739                        2                           84   
2830740                        1                           31   
2830741                        2                          192   
2830742                        2                          188   

          Total Length of Bwd Packets   Fwd Packet Length Max  \
0                                   0                       6   
1                                   6                       6   
2                                   6                       6   
3                                   6                       6   
4                                   0                       6   
...                               ...                     ...   
2830738                           152                      28   
2830739                           362                      42   
2830740                             6                      31   
2830741                           256                      32   
2830742                           226                      47   

          Fwd Packet Length Min   Fwd Packet Length Mean  \
0                             6                      6.0   
1                             6                      6.0   
2                             6                      6.0   
3                             6                      6.0   
4                             6                      6.0   
...                         ...                      ...   
2830738                      28                     28.0   
2830739                      42                     42.0   
2830740                       0                     15.5   
2830741                      32                     32.0   
2830742                      47                     47.0   

          Fwd Packet Length Std  ...   min_seg_size_forward  Active Mean  \
0                       0.00000  ...                     20          0.0   
1                       0.00000  ...                     20          0.0   
2                       0.00000  ...                     20          0.0   
3                       0.00000  ...                     20          0.0   
4                       0.00000  ...                     20          0.0   
...                         ...  ...                    ...          ...   
2830738                 0.00000  ...                     20          0.0   
2830739                 0.00000  ...                     20          0.0   
2830740                21.92031  ...                     32          0.0   
2830741                 0.00000  ...                     20          0.0   
2830742                 0.00000  ...                     2

In [46]:
data.columns.tolist()

[' Destination Port',
 ' Flow Duration',
 ' Total Fwd Packets',
 ' Total Backward Packets',
 'Total Length of Fwd Packets',
 ' Total Length of Bwd Packets',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Min',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std',
 'Bwd Packet Length Max',
 ' Bwd Packet Length Min',
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',
 'Flow Bytes/s',
 ' Flow Packets/s',
 ' Flow IAT Mean',
 ' Flow IAT Std',
 ' Flow IAT Max',
 ' Flow IAT Min',
 'Fwd IAT Total',
 ' Fwd IAT Mean',
 ' Fwd IAT Std',
 ' Fwd IAT Max',
 ' Fwd IAT Min',
 'Bwd IAT Total',
 ' Bwd IAT Mean',
 ' Bwd IAT Std',
 ' Bwd IAT Max',
 ' Bwd IAT Min',
 'Fwd PSH Flags',
 ' Bwd PSH Flags',
 ' Fwd URG Flags',
 ' Bwd URG Flags',
 ' Fwd Header Length',
 ' Bwd Header Length',
 'Fwd Packets/s',
 ' Bwd Packets/s',
 ' Min Packet Length',
 ' Max Packet Length',
 ' Packet Length Mean',
 ' Packet Length Std',
 ' Packet Length Variance',
 'FIN Flag Count',
 ' SYN Flag Count',
 ' RST Flag Count',
 ' P

In [47]:
data.columns[-1]

' Label'

In [48]:
group_map = {
    'PortScan': [
        "netAccess(machine2,tcp,22)", "netAccess(machine3,tcp,22)",
        "netAccess(machine3,tcp,53)", "netAccess(machine1,tcp,445)"
    ],
    'SSH-Patator': [
        "logInService(machine3,tcp,22)", "logInService(machine2,tcp,22)",
        "principalCompromised(victim)"
    ],
    'FTP-Patator': [
        "canAccessHost(machine1)", "canAccessHost(machine2)", "canAccessHost(machine3)"
    ],
    'DDoS': [
        "execCode(machine3,root)", "execCode(machine3,user)"
    ],
    'DoS GoldenEye': [
        "execCode(machine1,root)", "execCode(machine1,user)"
    ],
    'DoS Slowhttptest': [
        "execCode(machine2,user)", "execCode(machine2,root)"
    ]
}

group_to_keyword = {g: g for g in group_map}

desc_to_group = {}
for group_name, desc_list in group_map.items():
    for desc in desc_list:
        desc_to_group[desc] = group_name

node_desc_map = {
    i: f"{node_dict[i]['predicate']}({','.join(node_dict[i]['attributes'])})"
    for i in node_dict
}

attack_keywords = []
for idx in action_node_idx:
    desc = node_desc_map[idx]
    group = desc_to_group[desc]
    attack_keywords.append(group_to_keyword[group])

print(attack_keywords)
print(len(attack_keywords))

from collections import Counter
print(Counter(attack_keywords))

['FTP-Patator', 'DDoS', 'SSH-Patator', 'DoS GoldenEye', 'PortScan', 'DoS Slowhttptest', 'DoS Slowhttptest', 'FTP-Patator', 'PortScan', 'DoS GoldenEye', 'FTP-Patator', 'SSH-Patator', 'PortScan', 'SSH-Patator']
14
Counter({'FTP-Patator': 3, 'SSH-Patator': 3, 'PortScan': 3, 'DoS GoldenEye': 2, 'DoS Slowhttptest': 2, 'DDoS': 1})


In [49]:
g = data.groupby(' Label')[' Destination Port']
print(g.agg(lambda s: s.value_counts().index[0]))
print(g.agg(lambda s: s.value_counts(normalize=True).iloc[0]))

 Label
BENIGN                          53
Bot                           8080
DDoS                            80
DoS GoldenEye                   80
DoS Hulk                        80
DoS Slowhttptest                80
DoS slowloris                   80
FTP-Patator                     21
Heartbleed                     444
Infiltration                   444
PortScan                        80
SSH-Patator                     22
Web Attack � Brute Force        80
Web Attack � Sql Injection      80
Web Attack � XSS                80
Name:  Destination Port, dtype: int64
 Label
BENIGN                        0.421369
Bot                           0.641404
DDoS                          0.999977
DoS GoldenEye                 1.000000
DoS Hulk                      1.000000
DoS Slowhttptest              1.000000
DoS slowloris                 1.000000
FTP-Patator                   0.999874
Heartbleed                    1.000000
Infiltration                  1.000000
PortScan                      0.0

In [50]:
print(data[data[' Label']=='BENIGN'][' Destination Port'].value_counts().head(10))

 Destination Port
53     957812
443    505470
80     235695
123     23879
22      10801
137      7917
389      6247
88       5421
21       5341
465      3657
Name: count, dtype: int64


In [51]:
data = data.drop(columns=[' Destination Port'], errors='ignore')
assert ' Destination Port' not in data.columns
print(data.shape)

(2830743, 78)


In [52]:
x = data.iloc[:, :-1].values
y = data[' Label'].values

In [53]:
x.shape

(2830743, 77)

In [54]:
y.shape

(2830743,)

In [55]:
x=np.where(np.isinf(x),np.nan,x)
mask=~np.isnan(x).any(axis=1)
x=x[mask]
y=y[mask]

perm = np.random.RandomState(42).permutation(len(x))
x = x[perm]
y = y[perm]
print("shuffled:", x.shape)

shuffled: (2827876, 77)


In [56]:
x.shape

(2827876, 77)

In [57]:
y.shape

(2827876,)

In [59]:
num_action_nodes=len(action_node_idx)
num_benign=2400
num_malic=900

In [60]:
total_benign=num_benign*num_action_nodes+num_malic*num_action_nodes*num_action_nodes

In [61]:
total_benign

210000

In [62]:
benign_mask=(y=="BENIGN")
benign_rows=x[benign_mask]
benign_rows.shape[0]

2271320

In [63]:
benign_rows=benign_rows[:total_benign]
benign=benign_rows.reshape(-1,num_action_nodes,x.shape[1])

In [64]:
benign.shape

(15000, 14, 77)

In [65]:
x_benign=benign[:num_benign].copy()
y_benign=np.zeros((x_benign.shape[0],x_benign.shape[1]))

In [66]:
x_benign.shape

(2400, 14, 77)

In [67]:
y_benign.shape

(2400, 14)

In [68]:
x_malic=benign[num_benign:num_benign+num_malic*num_action_nodes].copy()
y_malic=np.zeros((x_malic.shape[0],x_malic.shape[1]))

In [69]:
x_malic.shape

(12600, 14, 77)

In [70]:
y_malic.shape

(12600, 14)

In [71]:
used_indices = {}

for idx in range(num_action_nodes):
    keyword = attack_keywords[idx]
    attack_mask = np.array([keyword in label for label in y])
    attack_rows = x[attack_mask]

    start_offset = used_indices.get(keyword, 0)
    end_offset = start_offset + num_malic

    print(f"Node {idx}: keyword='{keyword}', total available={attack_rows.shape[0]}, "
          f"taking rows[{start_offset}:{end_offset}]")

    if end_offset > attack_rows.shape[0]:
        print(f"  WARNING: sirf {attack_rows.shape[0]} rows hain, {end_offset} chahiye — SKIP karna padega")
        continue

    x_malic[idx*num_malic:(idx+1)*num_malic, idx, :] = attack_rows[start_offset:end_offset]
    used_indices[keyword] = end_offset
    y_malic[idx*num_malic:(idx+1)*num_malic, idx] = 1

print("\nDone. Final used_indices per keyword:", used_indices)

Node 0: keyword='FTP-Patator', total available=7935, taking rows[0:900]
Node 1: keyword='DDoS', total available=128025, taking rows[0:900]
Node 2: keyword='SSH-Patator', total available=5897, taking rows[0:900]
Node 3: keyword='DoS GoldenEye', total available=10293, taking rows[0:900]
Node 4: keyword='PortScan', total available=158804, taking rows[0:900]
Node 5: keyword='DoS Slowhttptest', total available=5499, taking rows[0:900]
Node 6: keyword='DoS Slowhttptest', total available=5499, taking rows[900:1800]
Node 7: keyword='FTP-Patator', total available=7935, taking rows[900:1800]
Node 8: keyword='PortScan', total available=158804, taking rows[900:1800]
Node 9: keyword='DoS GoldenEye', total available=10293, taking rows[900:1800]
Node 10: keyword='FTP-Patator', total available=7935, taking rows[1800:2700]
Node 11: keyword='SSH-Patator', total available=5897, taking rows[900:1800]
Node 12: keyword='PortScan', total available=158804, taking rows[1800:2700]
Node 13: keyword='SSH-Patator'

In [72]:
x_malic.shape

(12600, 14, 77)

In [73]:
y_malic.shape

(12600, 14)

In [74]:
y_malic.sum()

np.float64(12600.0)

In [75]:
y_malic

array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(12600, 14))

In [76]:
num_nodes=len(nodes)

In [77]:
rt_meas_dim=x_benign.shape[2]

In [78]:
rt_meas_dim

77

In [79]:
X_benign=np.zeros((num_benign,num_nodes,rt_meas_dim))
X_benign[:, action_node_idx, :]=x_benign
Y_benign=y_benign

In [80]:
X_malic=np.zeros((num_malic*num_action_nodes,num_nodes,rt_meas_dim))
X_malic[:,action_node_idx,:]=x_malic
Y_malic=y_malic

In [81]:
X=np.concatenate((X_benign,X_malic),axis=0)
Y=np.concatenate((Y_benign,Y_malic),axis=0)

In [82]:
X.shape

(15000, 51, 77)

In [83]:
Y.shape

(15000, 14)

In [84]:
print(num_action_nodes)
print(X.shape)
print(Y.shape)
print(y_malic.sum())

14
(15000, 51, 77)
(15000, 14)
12600.0


### Node Features

In [85]:
num_samples=X.shape[0]

In [86]:
node_feat_ts = np.stack([node_features for _ in range(num_samples)], axis=0)

In [87]:
node_feat_ts.shape

(15000, 51, 73)

In [88]:
X=np.concatenate((node_feat_ts,X),axis=2)

In [89]:
X.shape

(15000, 51, 150)

In [90]:
Y.shape

(15000, 14)

### Dataset Split

In [91]:
X_t=torch.tensor(X,dtype=torch.float32)
Y_t=torch.tensor(Y,dtype=torch.float32)
edge_index_t=torch.tensor(edge_index,dtype=torch.long)

In [92]:
X_train,X_test,Y_train,Y_test=train_test_split(X_t,Y_t,test_size=0.2,random_state=42,stratify=Y_t)

In [93]:
X_train,X_val,Y_train,Y_val=train_test_split(X_train,Y_train,test_size=0.25,random_state=42,stratify=Y_train)

### safe scaling 

In [94]:
from sklearn.metrics import f1_score as _f1, precision_score as _prec, recall_score as _rec
node_feat_dim = node_features.shape[1]

scaler = MinMaxScaler(feature_range=(0, 1))
_tr = X_train[:, action_node_idx, node_feat_dim:]
scaler.fit(_tr.reshape(-1, _tr.shape[-1]).numpy())

def apply_scaler(X):
    X = X.clone()
    sub = X[:, action_node_idx, node_feat_dim:]
    shp = sub.shape
    X[:, action_node_idx, node_feat_dim:] = torch.tensor(
        scaler.transform(sub.reshape(-1, shp[-1]).numpy()), dtype=torch.float32).reshape(shp)
    return X

X_train, X_val, X_test = apply_scaler(X_train), apply_scaler(X_val), apply_scaler(X_test)
print("scaler fit on", _tr.shape[0] * _tr.shape[1], "TRAIN traffic rows; applied to train / val / test")

scaler fit on 126000 TRAIN traffic rows; applied to train / val / test


In [95]:
X_train.shape

torch.Size([9000, 51, 150])

In [96]:
Y_train.shape

torch.Size([9000, 14])

In [97]:
X_val.shape

torch.Size([3000, 51, 150])

In [98]:
Y_val.shape

torch.Size([3000, 14])

In [99]:
X_test.shape

torch.Size([3000, 51, 150])

In [100]:
Y_test.shape

torch.Size([3000, 14])

In [101]:
edge_index_t.shape

torch.Size([2, 72])

### GCN-EW Model — Multi-Seed Training

In [102]:
in_dim_ew = X_train.shape[-1]
hidden_dim = 20
out_dim = 1

num_epoch = 150
seeds = [42, 123, 999, 7, 2024]

In [103]:
def train_and_evaluate_gcn_ew(seed):

    torch.manual_seed(seed)

    edges = edge_index_t.to(device)
    model_ew = GCN_EW(in_dim_ew, hidden_dim, out_dim, edges).to(device)

    X_train_dev = X_train.to(device)
    Y_train_dev = Y_train.to(device)
    X_val_dev = X_val.to(device)
    Y_val_dev = Y_val.to(device)

    num_class_0_ew = (Y_train_dev == 0).sum().item()
    num_class_1_ew = (Y_train_dev == 1).sum().item()
    pos_weight_ew = torch.tensor([num_class_0_ew / num_class_1_ew], dtype=torch.float32).to(device)
    criterion_ew = nn.BCEWithLogitsLoss(pos_weight=pos_weight_ew)

    optimizer_ew = optim.Adam(model_ew.parameters(), lr=0.001)

    def evaluate_ew(X_data, Y_data):
        model_ew.eval()
        with torch.no_grad():
            output = model_ew(X_data, edges)
            loss = criterion_ew(output[:, action_node_idx], Y_data)
            pred = (torch.sigmoid(output[:, action_node_idx]) > 0.5).float()
            accuracy = (pred == Y_data).sum().item() / Y_data.numel()
        return loss.item(), accuracy

    best_val_loss = float('inf')
    best_state = None

    train_dataset = TensorDataset(X_train_dev, Y_train_dev)
    train_loader = DataLoader(train_dataset, batch_size=30, shuffle=False)

    for epoch in range(num_epoch):
        model_ew.train()

        for batch_X, batch_Y in train_loader:
            output = model_ew(batch_X, edges)
            loss = criterion_ew(output[:, action_node_idx], batch_Y)

            optimizer_ew.zero_grad()
            loss.backward()
            optimizer_ew.step()

        val_loss, val_acc = evaluate_ew(X_val_dev, Y_val_dev)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model_ew.state_dict().items()}

    model_ew.load_state_dict(best_state)
    model_ew.eval()

    torch.save(model_ew.state_dict(), f"best_ew_fixed_seed{seed}.pt")

    all_preds_ew, all_probs_ew, all_labels_ew = [], [], []

    with torch.no_grad():
        for i in range(len(X_test)):
            x_i = X_test[i].to(device)
            y_i = Y_test[i].to(device)

            output = model_ew(x_i, edges)
            probs = torch.sigmoid(output[action_node_idx])
            preds = (probs > 0.5).float()

            all_probs_ew.append(probs.cpu())
            all_preds_ew.append(preds.cpu())
            all_labels_ew.append(y_i.cpu())

    all_probs_ew = torch.cat(all_probs_ew).numpy()
    all_preds_ew = torch.cat(all_preds_ew).numpy()
    all_labels_ew = torch.cat(all_labels_ew).numpy()

    cm_ew = confusion_matrix(all_labels_ew, all_preds_ew)
    TN, FP, FN, TP = cm_ew.ravel()

    accuracy_ew = accuracy_score(all_labels_ew, all_preds_ew)
    precision_ew = precision_score(all_labels_ew, all_preds_ew, zero_division=0)
    recall_ew = recall_score(all_labels_ew, all_preds_ew, zero_division=0)
    f1_ew = f1_score(all_labels_ew, all_preds_ew, zero_division=0)
    fpr_ew, tpr_ew, _ = roc_curve(all_labels_ew, all_probs_ew)
    roc_auc_ew = auc(fpr_ew, tpr_ew)

    return {
        'seed': seed,
        'accuracy': accuracy_ew,
        'precision': precision_ew,
        'recall': recall_ew,
        'f1': f1_ew,
        'auc': roc_auc_ew,
        'fpr': FP / (FP + TN),
        'fnr': FN / (FN + TP),
        'precision_macro': _prec(all_labels_ew, all_preds_ew, average='macro', zero_division=0),
        'recall_macro': _rec(all_labels_ew, all_preds_ew, average='macro', zero_division=0),
        'f1_macro': _f1(all_labels_ew, all_preds_ew, average='macro', zero_division=0),
    }

In [104]:
results = []

for seed in seeds:
    print(f"--- Training with seed {seed} ---")
    metrics = train_and_evaluate_gcn_ew(seed)
    results.append(metrics)
    print(metrics)
    print()

--- Training with seed 42 ---
{'seed': 42, 'accuracy': 0.9854047619047619, 'precision': 0.8059031119666346, 'recall': 0.9968253968253968, 'f1': 0.8912542132339897, 'auc': np.float64(0.998281664616201), 'fpr': np.float64(0.015324214792299898), 'fnr': np.float64(0.0031746031746031746), 'precision_macro': 0.902848683262591, 'recall_macro': 0.9907505910165484, 'f1_macro': 0.9417158219545905}

--- Training with seed 123 ---
{'seed': 123, 'accuracy': 0.9772857142857143, 'precision': 0.7269565217391304, 'recall': 0.9952380952380953, 'f1': 0.8402010050251256, 'auc': np.float64(0.9973265044788602), 'fpr': np.float64(0.02386018237082067), 'fnr': np.float64(0.004761904761904762), 'precision_macro': 0.8633226188462189, 'recall_macro': 0.9856889564336373, 'f1_macro': 0.9139874690638892}

--- Training with seed 999 ---
{'seed': 999, 'accuracy': 0.978047619047619, 'precision': 0.7333528037383178, 'recall': 0.9964285714285714, 'f1': 0.844885598923284, 'auc': np.float64(0.9977963576092377), 'fpr': np.f

### Multi-Seed Summary

In [105]:
results_df = pd.DataFrame(results)
results_df

,seed,accuracy,precision,recall,f1,auc,fpr,fnr,precision_macro,recall_macro,f1_macro
0,42,0.985405,0.805903,0.996825,0.891254,0.998282,0.015324,0.003175,0.902849,0.990751,0.941716
1,123,0.977286,0.726957,0.995238,0.840201,0.997327,0.023860,0.004762,0.863323,0.985689,0.913987
2,999,0.978048,0.733353,0.996429,0.844886,0.997796,0.023126,0.003571,0.866560,0.986651,0.916537
3,7,0.977571,0.730432,0.992460,0.841521,0.997785,0.023379,0.007540,0.864970,0.984541,0.914726
4,2024,0.987429,0.828496,0.996825,0.904899,0.998519,0.013171,0.003175,0.914145,0.991827,0.949084


In [106]:
print("--- Mean ± Std across", len(seeds), "seeds ---")
for col in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr', 'fnr']:
    print(f"{col:10s}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}")

--- Mean ± Std across 5 seeds ---
accuracy  : 0.9811 ± 0.0049
precision : 0.7650 ± 0.0483
recall    : 0.9956 ± 0.0018
f1        : 0.8646 ± 0.0310
auc       : 0.9979 ± 0.0005
fpr       : 0.0198 ± 0.0051
fnr       : 0.0044 ± 0.0018


### Summary : attack-class metrics + macro for reference, saved to CSV

In [107]:
results_df = pd.DataFrame(results)
cols = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fpr', 'fnr', 'precision_macro', 'recall_macro', 'f1_macro']
print("--- GCN_EW (FIXED) | mean +- std across", len(seeds), "seeds | precision/recall/f1 = attack class ---")
for col in cols:
    print(f"{col:16s}: {results_df[col].mean():.4f} +- {results_df[col].std():.4f}")
results_df.to_csv("results_gcn_ew_fixed.csv", index=False)
print("\nSaved: results_gcn_ew_fixed.csv")

--- GCN_EW (FIXED) | mean +- std across 5 seeds | precision/recall/f1 = attack class ---
accuracy        : 0.9811 +- 0.0049
precision       : 0.7650 +- 0.0483
recall          : 0.9956 +- 0.0018
f1              : 0.8646 +- 0.0310
auc             : 0.9979 +- 0.0005
fpr             : 0.0198 +- 0.0051
fnr             : 0.0044 +- 0.0018
precision_macro : 0.8824 +- 0.0242
recall_macro    : 0.9879 +- 0.0032
f1_macro        : 0.9272 +- 0.0168

Saved: results_gcn_ew_fixed.csv
